In [10]:
import os
import requests
from pathlib import Path
from shapely.geometry import Point
import geopandas as gpd
from tqdm import tqdm
import shutil
import json
import subprocess

def generate_lidar_filenames(lat, lon, buffer_km=5):
    pt = gpd.GeoSeries([Point(lon, lat)], crs="EPSG:4326").to_crs(epsg=2154).geometry[0]
    x_center, y_center = pt.x, pt.y
    x_min = int((x_center - buffer_km * 1000) // 1000)
    x_max = int((x_center + buffer_km * 1000) // 1000)
    y_min = int((y_center - buffer_km * 1000) // 1000)
    y_max = int((y_center + buffer_km * 1000) // 1000)

    return [
        f"LHD_FXX_{x:04d}_{y:04d}_PTS_C_LAMB93_IGN69.copc.laz"
        for x in range(x_min, x_max + 1)
        for y in range(y_min, y_max + 1)
    ]

def download_if_missing(filenames, base_url, dest_dir):
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(exist_ok=True)

    for fname in tqdm(filenames, desc="📥 Downloading tiles"):
        url = base_url + fname
        dest_path = dest_dir / fname
        if dest_path.exists():
            continue
        try:
            response = requests.get(url, stream=True, timeout=30)
            if response.status_code == 404:
                print(f"⛔ Tile not found: {fname}")
                continue
            response.raise_for_status()
            with open(dest_path, "wb") as f_out:
                for chunk in response.iter_content(chunk_size=8192):
                    f_out.write(chunk)
        except Exception as e:
            print(f"⚠️ Failed to download {fname}: {e}")
            continue

def copy_selected_tiles(filenames, source_dir, target_dir):
    source_dir = Path(source_dir)
    target_dir = Path(target_dir)

    # Remove existing folder and recreate
    if target_dir.exists():
        shutil.rmtree(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)

    for fname in tqdm(filenames, desc="📤 Copying selected tiles"):
        src = source_dir / fname
        dst = target_dir / fname
        if src.exists():
            shutil.copy2(src, dst)
        else:
            print(f"⚠️ File not found in source: {fname}")

def main(lat, lon, name, buffer_km=3):
    base_url = "https://storage.sbg.cloud.ovh.net/v1/AUTH_63234f509d6048bca3c9fd7928720ca1/ppk-lidar/SE/"
    all_lidar_dir = "site_data/all_lidar"
    selected_lidar_dir = f"site_data/{name}/selected_lidar"

    print("🔍 Generating filenames...")
    filenames = generate_lidar_filenames(lat, lon, buffer_km)

    print("⬇️ Downloading missing tiles...")
    download_if_missing(filenames, base_url, all_lidar_dir)

    print("📁 Copying selected tiles to output folder...")
    copy_selected_tiles(filenames, all_lidar_dir, selected_lidar_dir)


    

    # Define the pipeline dictionary
    pipeline = {
        "pipeline": [
            f"site_data/{name}/selected_lidar/*.laz",
            {
            "type": "writers.gdal",
            "filename": f"site_data/{name}/dsm_max.tif",
            "resolution": 1.0,
            "output_type": "max"
            },
            {
            "type": "writers.gdal",
            "filename": f"site_data/{name}/dsm_min.tif",
            "resolution": 1.0,
            "output_type": "min"
            }
        ]
    }

    # Save to a JSON file
    with open(f"site_data/{name}/pipeline_config.json", "w") as f:
        json.dump(pipeline, f, indent=2)

    print("📁 To Create tif file please run : pdal pipeline", f"site_data/{name}/pipeline_config.json")

    print("✅ Done!")

In [8]:
#main(lat=48.379147749213516	, lon=2.821026152099335, name="moret", buffer_km=3)
#main(lat=48.426777064786336    ,lon=2.710721365846438, name="augas", buffer_km=3)
#main(lat=48.26050384702055 	,lon=2.706470479996845, name="nemours", buffer_km=3)
#main(lat=48.478465901909985	,lon=2.424674041753525, name="videlles", buffer_km=3)

In [11]:
main(lat=48.8221944	,lon=7.7798889, name="haguenau", buffer_km=3)
# main(lat=48.8974722	,lon=7.9106944, name="betschdorf", buffer_km=3)
# main(lat=48.7723611	,lon=7.8465, name="bischwiller", buffer_km=3)

🔍 Generating filenames...
⬇️ Downloading missing tiles...


📥 Downloading tiles: 100%|██████████| 49/49 [00:00<00:00, 76686.90it/s]


📁 Copying selected tiles to output folder...


📤 Copying selected tiles: 100%|██████████| 49/49 [00:02<00:00, 17.36it/s]

📁 To Create tif file please run : pdal pipeline site_data/haguenau/pipeline_config.json
✅ Done!


In [4]:
48.8184, 7.7912

(48.8184, 7.7912)